In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/vikramtriplicane/penicillin/100_Batches_IndPenSim_V3.csv


In [4]:
# =========================================================
# PENICILLIN YIELD PREDICTION USING AN ARTIFICIAL NEURAL NETWORK (ANN)
# =========================================================
# This script:
# 1. Loads the penicillin production dataset
# 2. Cleans it (drops fully-empty columns)
# 3. Selects the 5 key reaction condition features
# 4. Splits data by BATCH ID (not by row) to prevent data leakage between
#    time points of the same batch
# 5. Scales features and target (ANNs are sensitive to input scale)
# 6. Builds and trains a feedforward neural network
# 7. Evaluates the model using RMSE, MAE, and R²
# 8. Lets the user input their own reaction conditions to predict yield
# =========================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# ---------------------------------------------------------
# STEP 1: Load the dataset
# ---------------------------------------------------------
file_path = "/kaggle/input/datasets/vikramtriplicane/penicillin/100_Batches_IndPenSim_V3.csv"
df = pd.read_csv(file_path)

# ---------------------------------------------------------
# STEP 2: Clean the dataset
# ---------------------------------------------------------
# Columns 201 and 202 (by position) contain only missing values per the EDA,
# so we drop them if present.
cols_to_check = df.columns[[201, 202]] if df.shape[1] > 202 else []
df = df.drop(columns=[c for c in cols_to_check if df[c].isnull().all()], errors="ignore")

# =========================================================
# DIAGNOSTIC: find the column that actually represents the 100 batches
# =========================================================
# Run this in a new cell in your Kaggle notebook, after df is loaded
# (i.e. after your existing Cell 1's file-loading step).
#
# We know from the dataset's own documentation that there should be
# exactly 100 batches. This checks EVERY column's number of unique
# values and shows us which ones are plausible candidates (somewhere
# close to 100), so we can stop guessing from column names alone.

print(f"Total rows in dataset: {len(df)}")
print(f"Total columns: {len(df.columns)}\n")

print("Columns with between 2 and 500 unique values (batch ID candidates):")
for col in df.columns:
    try:
        n_unique = df[col].nunique()
        if 2 <= n_unique <= 500:
            print(f"  {col!r}: {n_unique} unique values")
    except TypeError:
        # some columns (e.g. containing unhashable types) can't be checked this way
        continue

# =========================================================
# CONFIRM: which of the two candidate columns is the real batch column?
# =========================================================
# Both '2-PAT control(PAT_ref:PAT ref)' and ' 1-Raman spec recorded'
# have exactly 100 unique values. A real batch column should group rows
# into ~100 chunks that are each a few hundred to a couple thousand rows
# (since 113,935 rows / 100 batches \u2248 1,139 rows per batch on average).
# Whichever candidate shows that pattern is the real one.

for col in ["2-PAT control(PAT_ref:PAT ref)", " 1-Raman spec recorded"]:
    print(f"\n--- {col!r} ---")
    counts = df[col].value_counts()
    print(f"Number of groups: {counts.shape[0]}")
    print(f"Rows per group -> min: {counts.min()}, max: {counts.max()}, mean: {counts.mean():.0f}")
    print(f"Sum of all group sizes: {counts.sum()} (should equal {len(df)})")
    
# ---------------------------------------------------------
# STEP 3: Set the batch identifier column
# ---------------------------------------------------------
# The dataset contains two columns with "batch" in the name:
# 'Batch reference(Batch_ref:Batch ref)' (only 2 unique values — not useful
# for splitting) and 'Batch ID' (the actual per-batch identifier, ~100
# unique values matching the "100 Batches" dataset).
batch_column = "2-PAT control(PAT_ref:PAT ref)"

# Sanity check: confirm this column has a reasonable number of unique
# batches (should be around 100).
print(f"Unique values in '{batch_column}': {df[batch_column].nunique()}")

# ---------------------------------------------------------
# STEP 4: Define required columns and drop missing rows
# ---------------------------------------------------------
required_columns = [
    batch_column,
    "Substrate concentration(S:g/L)",
    "Dissolved oxygen concentration(DO2:mg/L)",
    "pH(pH:pH)",
    "Temperature(T:K)",
    "Time (h)",
    "Penicillin concentration(P:g/L)"
]
df = df.dropna(subset=required_columns)

feature_columns = [
    "Substrate concentration(S:g/L)",
    "Dissolved oxygen concentration(DO2:mg/L)",
    "pH(pH:pH)",
    "Temperature(T:K)",
    "Time (h)"
]
target_column = "Penicillin concentration(P:g/L)"

# ---------------------------------------------------------
# STEP 5: Split by BATCH, not by row
# ---------------------------------------------------------
# Get the unique batch identifiers first, then split those into train/test
# groups. Only after that do we pull out all the rows belonging to each
# group. This ensures no batch has some hours in training and other hours
# in test — entire batches are held out.
unique_batches = df[batch_column].unique()
print(f"Total unique batches found: {len(unique_batches)}")

train_batches, test_batches = train_test_split(
    unique_batches, test_size=0.2, random_state=42
)

train_df = df[df[batch_column].isin(train_batches)]
test_df = df[df[batch_column].isin(test_batches)]

print(f"Training batches: {len(train_batches)} ({len(train_df)} rows)")
print(f"Test batches:     {len(test_batches)} ({len(test_df)} rows)")

X_train = train_df[feature_columns]
y_train = train_df[target_column]
X_test = test_df[feature_columns]
y_test = test_df[target_column]

# ---------------------------------------------------------
# STEP 6: Feature and target scaling
# ---------------------------------------------------------
# ANNs train much better when inputs are on a similar scale (unlike Random
# Forests, which don't care about scale). We use StandardScaler to
# standardize both the input features and the target variable.
# Fit the scaler ONLY on training data to avoid leaking test information.
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

# ---------------------------------------------------------
# STEP 7: Build the ANN architecture
# ---------------------------------------------------------
# A feedforward network:
# - Input layer matches the 5 features
# - Three hidden layers with ReLU activation to learn nonlinear relationships
# - Dropout layers to reduce overfitting
# - Single output neuron for the regression target (yield)
model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu"),
    layers.Dense(1)  # output layer: predicted (scaled) penicillin concentration
])

# Compile the model: Adam optimizer, mean squared error as the loss function
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="mse",
    metrics=["mae"]
)

model.summary()

# ---------------------------------------------------------
# STEP 8: Train the ANN
# ---------------------------------------------------------
# NOTE: validation_split here still splits the TRAINING rows randomly,
# which means some validation leakage across batches can still occur
# during training (used only for early stopping, not final evaluation).
# The final performance numbers in Step 9 come from the fully held-out
# test batches, which is the leakage-free comparison that matters.
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=30,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train_scaled,
    validation_split=0.2,
    epochs=200,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

print(f"\nTraining stopped after {len(history.history['loss'])} epochs")


Total rows in dataset: 113935
Total columns: 2239

Columns with between 2 and 500 unique values (batch ID candidates):
  'Aeration rate(Fg:L/h)': 7 unique values
  'Sugar feed rate(Fs:L/h)': 25 unique values
  'Water for injection/dilution(Fw:L/h)': 6 unique values
  'Air head pressure(pressure:bar)': 6 unique values
  'Dumped broth flow(Fremoved:L/h)': 2 unique values
  'Temperature(T:K)': 393 unique values
  'Oil flow(Foil:L/hr)': 9 unique values
  'Fault reference(Fault_ref:Fault ref)': 2 unique values
  '0 - Recipe driven 1 - Operator controlled(Control_ref:Control ref)': 2 unique values
  '1- No Raman spec': 2 unique values
  ' 1-Raman spec recorded': 100 unique values
  '2-PAT control(PAT_ref:PAT ref)': 100 unique values
  'Batch reference(Batch_ref:Batch ref)': 2 unique values

--- '2-PAT control(PAT_ref:PAT ref)' ---
Number of groups: 100
Rows per group -> min: 835, max: 1450, mean: 1139
Sum of all group sizes: 113935 (should equal 113935)

--- ' 1-Raman spec recorded' ---
Numb

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 256)            │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 42,753 (167.00 KB)

 Trainable params: 42,753 (167.00 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/200
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 8s 6ms/step - loss: 0.0391 - mae: 0.1406 - val_loss: 0.0311 - val_mae: 0.1404
Epoch 2/200
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.0223 - mae: 0.1120 - val_loss: 0.0439 - val_mae: 0.1674
Epoch 3/200
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.0197 - mae: 0.1043 - val_loss: 0.0355 - val_mae: 0.1512
Epoch 4/200
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.0183 - mae: 0.0996 - val_loss: 0.0461 - val_mae: 0.1796
Epoch 5/200
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.0173 - mae: 0.0964 - val_loss: 0.0567 - val_mae: 0.2006
Epoch 6/200
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.0168 - mae: 0.0946 - val_loss: 0.0424 - val_mae: 0.1688
Epoch 7/200
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 6s 5ms/step - loss: 0.0162 - mae: 0.0928 - val_loss: 0.0479 - val_mae: 0.1782
Epoch 8/200
1141/1141 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - loss: 0.0159 - mae: 0.0915 - val_loss: 0.0361 - val_mae: 0.1556
Epoch 9/200
1141/1141 ━━━━━━━━━

In [5]:
# ---------------------------------------------------------
# STEP 9: Evaluate the model on held-out batches
# ---------------------------------------------------------
y_pred_scaled = model.predict(X_test_scaled)
y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()
y_test_actual = y_test.values

rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
mae = mean_absolute_error(y_test_actual, y_pred)
r2 = r2_score(y_test_actual, y_pred)

print("\nModel Performance on Held-Out Test Batches:")
print(f"RMSE (Root Mean Square Error): {rmse:.4f}")
print(f"MAE  (Mean Absolute Error):    {mae:.4f}")
print(f"R²   (Coefficient of Determination): {r2:.4f}")

# ---------------------------------------------------------
# STEP 10: Predict yield from user-input reaction conditions
# ---------------------------------------------------------
def get_user_prediction(model, scaler_X, scaler_y):
    print("\nEnter your reaction conditions to predict penicillin yield:")

    substrate_conc = float(input("Substrate concentration (S: g/L): "))
    dissolved_o2 = float(input("Dissolved oxygen concentration (DO2: mg/L): "))
    ph_level = float(input("pH: "))
    temperature = float(input("Temperature (T: K): "))
    time_h = float(input("Time (h): "))

    user_input_df = pd.DataFrame([{
        "Substrate concentration(S:g/L)": substrate_conc,
        "Dissolved oxygen concentration(DO2:mg/L)": dissolved_o2,
        "pH(pH:pH)": ph_level,
        "Temperature(T:K)": temperature,
        "Time (h)": time_h
    }])

    user_input_scaled = scaler_X.transform(user_input_df)
    predicted_scaled = model.predict(user_input_scaled)
    predicted_yield = scaler_y.inverse_transform(predicted_scaled)[0][0]

    print(f"\nPredicted Penicillin Concentration (P: g/L): {predicted_yield:.4f}")
    return predicted_yield

get_user_prediction(model, scaler_X, scaler_y)

710/710 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

Model Performance on Held-Out Test Batches:
RMSE (Root Mean Square Error): 1.7606
MAE  (Mean Absolute Error):    1.3068
R²   (Coefficient of Determination): 0.9719

Enter your reaction conditions to predict penicillin yield:


Substrate concentration (S: g/L):  35
Dissolved oxygen concentration (DO2: mg/L):  45
pH:  8
Temperature (T: K):  200
Time (h):  9


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step

Predicted Penicillin Concentration (P: g/L): 65.4784


np.float32(65.47838)

In [6]:
import joblib

model.save("penicillin_ann_model.keras")
joblib.dump(scaler_X, "scaler_X.pkl")
joblib.dump(scaler_y, "scaler_y.pkl")

print("Saved 3 files. Go to the 'Output' panel on the right side of the")
print("Kaggle notebook and download all three:")
print("  - penicillin_ann_model.keras")
print("  - scaler_X.pkl")
print("  - scaler_y.pkl")

Saved 3 files. Go to the 'Output' panel on the right side of the
Kaggle notebook and download all three:
  - penicillin_ann_model.keras
  - scaler_X.pkl
  - scaler_y.pkl
